# Notebook 17 — Cross-Dataset Transformer Validation of VoxIntel-R

## Why this notebook exists

Notebook 16 introduced the first reference-free VoxIntel-R experiment on SLURP. It tested
whether downstream intent failure can be predicted using only signals available at
inference time — ASR-native uncertainty and intent-model uncertainty — without access to
the reference transcript.

That experiment produced a useful but unresolved result: intent-model uncertainty was
strongly predictive of downstream failure, while adding ASR-native uncertainty features
did not clearly improve on the intent-only baseline. That result was obtained within a
single dataset and a single model family, and the Notebook 16 evaluation population still
needs a final fixed-split rerun before its numbers are treated as canonical.

This notebook asks a different question:

> Does the reference-free VoxIntel-R reliability signal generalize to a different
> spoken-language-understanding dataset and independently trained Transformer models?

We use **Fluent Speech Commands (FSC)** as an external validation dataset — 30,043 spoken
commands from 97 speakers across 31 intents, with official speaker-independent
train/validation/test splits.

The goal is **not** to search for a dataset on which H1 happens to succeed. The goal is to
test whether the same reference-free risk formulation survives a change in:

- dataset
- speakers
- command vocabulary
- acoustic conditions
- ASR model checkpoint
- intent Transformer checkpoint

The experiment reproduces the core VoxIntel-R protocol from Notebook 16 while keeping the
feature definitions and the leakage boundary fixed.

**Revision note (this version):** the notebook was previously a scaffold — the ASR and
intent fine-tuning functions raised `NotImplementedError`, `run_intent_inference` was
called but never defined, and most of the downstream pipeline (Sections 08, 11-18) was
commented out. This revision implements those pieces end-to-end: real CTC fine-tuning for
the FSC Wav2Vec2 checkpoint, a real `Trainer`-based fine-tuning loop for the FSC DistilBERT
intent checkpoint, a defined `run_intent_inference`, and feature generation on **both**
validation (for risk-model fitting) and test (for frozen evaluation) — not test alone.
Training is gated behind `RUN_ASR_TRAINING` / `RUN_INTENT_TRAINING` flags in Section 03 so
opening the notebook does not silently kick off multi-hour GPU jobs; every downstream cell
checks for the presence of a trained checkpoint and reports what it's waiting on rather
than failing or silently no-op'ing.

### Research hypothesis

**H1 — Cross-dataset complementarity**

> Reference-free ASR-native and intent-native uncertainty signals contain complementary
> information for predicting downstream intent failure, such that their combination
> improves risk prediction over intent uncertainty alone on an unseen spoken-command
> dataset.

### Critical anti-leakage rule

The reference transcript may be used to construct the supervised target and for final
evaluation, but it must **never** be used to construct an inference-time risk feature.

**Allowed:** ASR logits/probabilities · ASR hypothesis · audio duration · intent
logits/probabilities

**Forbidden:** reference transcript · WER/CER · reference/hypothesis alignment · lexical
overlap with the reference · reference-derived error taxonomy

The experiment is considered invalid if any forbidden signal enters the risk feature
matrix. Section 15 audits this automatically.

### Sequencing note

This notebook is inserted **between** Notebook 16 and the calibration notebook. The
finalized README originally scheduled calibration + selective prediction as Notebook 17;
this insertion is a deliberate research-sequencing decision (Notebook 16's SLURP result is
provisional and H1 is not yet supported there), not an accidental drift. The README's
roadmap section is updated separately, after this notebook produces an actual result —
not before.

    16  Reference-free VoxIntel-R on SLURP
            ↓
    17  Cross-dataset Transformer validation   ← this notebook
            ↓
    18  Calibration + selective prediction
            ↓
    19  Cost-sensitive selective prediction

## 01 — Research contract

Fixes the notebook's identity, dataset, hypothesis, and seed before any code that could
be influenced by them runs.

In [1]:
NOTEBOOK_ID = "17"
DATASET = "FSC"                 # Fluent Speech Commands — external validation dataset
HYPOTHESIS = "H1"                # Cross-dataset complementarity (see intro cell)
RANDOM_SEED = 42

import random
import numpy as np
import torch

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

print("=" * 50)
print(f"Notebook:        {NOTEBOOK_ID}")
print(f"Dataset:         Fluent Speech Commands")
print(f"Purpose:         external validation of VoxIntel-R (Notebook 16 protocol)")
print(f"Hypothesis:      {HYPOTHESIS} — cross-dataset complementarity")
print(f"Reference-free:  YES (see anti-leakage audit, Section 15)")
print(f"Random seed:     {RANDOM_SEED}")
print("=" * 50)

Notebook:        17
Dataset:         Fluent Speech Commands
Purpose:         external validation of VoxIntel-R (Notebook 16 protocol)
Hypothesis:      H1 — cross-dataset complementarity
Reference-free:  YES (see anti-leakage audit, Section 15)
Random seed:     42


## 02 — Imports

Reuses the existing project stack wherever the existing `src` modules are already generic
(the audio loader). Everything specific to reference-aware taxonomy machinery from
Notebooks 09–15 is deliberately **not** imported — this notebook is self-contained and
reference-free by construction.

In [2]:
from pathlib import Path
import sys
import json
import math
import random
import warnings

# ---------------------------------------------------------------------------
# PROJECT ROOT / IMPORT PATH
# ---------------------------------------------------------------------------
PROJECT_ROOT = Path(r"C:\Users\ACER\OneDrive\Desktop\VoxIntel").resolve()

if not (PROJECT_ROOT / "src").is_dir():
    raise FileNotFoundError(
        f"VoxIntel project root is invalid.\n"
        f"PROJECT_ROOT: {PROJECT_ROOT}\n"
        f"Expected: {PROJECT_ROOT / 'src'}"
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")
print(f"src exists:   {(PROJECT_ROOT / 'src').is_dir()}")

import numpy as np
import pandas as pd

import torch
import torch.nn.functional as F
import torchaudio

from transformers import (
    AutoProcessor,
    Wav2Vec2ForCTC,
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    confusion_matrix,
)

import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

from src.utils.audio import load_audio

print("Imports OK.")

Project root: C:\Users\ACER\OneDrive\Desktop\VoxIntel
src exists:   True


c:\Users\ACER\anaconda3\envs\torch26\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Imports OK.


## 03 — Configuration

Locks every path up front. FSC's official release layout is:

    data/
        train_data.csv
        valid_data.csv
        test_data.csv
    wavs/
        speakers/

which is kept separate from the SLURP paths used elsewhere in the project.

In [3]:
# --- FSC dataset paths ------------------------------------------------------
FSC_ROOT = PROJECT_ROOT / "data" / "raw" / "fsc" / "fluent_speech_commands_dataset"
FSC_DATA_DIR = FSC_ROOT / "data"
FSC_AUDIO_DIR = FSC_ROOT / "wavs"

FSC_TRAIN_CSV = FSC_DATA_DIR / "train_data.csv"
FSC_VALID_CSV = FSC_DATA_DIR / "valid_data.csv"
FSC_TEST_CSV = FSC_DATA_DIR / "test_data.csv"

# --- Model output paths ------------------------------------------------------
MODEL_ROOT = PROJECT_ROOT / "models"
FSC_ASR_MODEL_DIR = MODEL_ROOT / "wav2vec2_fsc"
FSC_INTENT_MODEL_DIR = MODEL_ROOT / "distilbert_fsc_intent"

# Base checkpoints — same model families as the original SLURP pipeline
# (Notebooks 03-06 / 07), so the cross-dataset comparison stays interpretable.
# Explicitly NOT the SLURP fine-tuned checkpoints: FSC needs its own.
BASE_ASR_MODEL = "facebook/wav2vec2-base-960h"
BASE_INTENT_MODEL = "distilbert-base-uncased"

# --- Report / artifact paths -------------------------------------------------
REPORT_DIR = PROJECT_ROOT / "reports"

FSC_AUDIT_PATH = REPORT_DIR / "fsc_dataset_audit.csv"
FSC_ASR_PRED_PATH = REPORT_DIR / "fsc_asr_predictions.csv"
FSC_INTENT_PRED_PATH = REPORT_DIR / "fsc_intent_predictions.csv"
FEATURE_PATH = REPORT_DIR / "fsc_voxintel_r_features.csv"
RESULTS_PATH = REPORT_DIR / "fsc_voxintel_r_model_comparison.csv"
PREDICTIONS_PATH = REPORT_DIR / "fsc_voxintel_r_predictions.csv"
IMPORTANCE_PATH = REPORT_DIR / "fsc_voxintel_r_feature_importance.csv"
CROSS_DATASET_PATH = REPORT_DIR / "fsc_voxintel_r_cross_dataset_comparison.csv"
SUMMARY_PATH = REPORT_DIR / "fsc_voxintel_r_summary.json"

for p in [MODEL_ROOT, REPORT_DIR, FSC_ASR_MODEL_DIR, FSC_INTENT_MODEL_DIR]:
    p.mkdir(parents=True, exist_ok=True)

# --- Execution gates ----------------------------------------------------------
# Training is expensive (GPU, ~30k utterances). Opening/running this notebook
# top-to-bottom must NOT silently kick off multi-hour training. Flip these to
# True only when you intend to actually fine-tune that checkpoint.
RUN_ASR_TRAINING = True
RUN_INTENT_TRAINING = True


def _dir_has_checkpoint(path: Path) -> bool:
    """True if `path` exists and contains a saved HF checkpoint."""
    if not path.exists():
        return False
    return any(path.iterdir())


# --- The exact Notebook 16 reference-free feature schema (must match) -------
ASR_FEATURES = [
    "asr_mean_confidence",
    "asr_min_confidence",
    "asr_std_confidence",
    "asr_median_confidence",
    "asr_mean_entropy",
    "asr_max_entropy",
    "asr_std_entropy",
    "asr_num_frames",
    "audio_duration",
]

INTENT_FEATURES = [
    "intent_confidence",
    "intent_entropy",
    "intent_margin",
]

FORBIDDEN_FEATURES = {
    "ground_truth_intent",
    "reference_transcript",
    "wer",
    "cer",
    "taxonomy",
    "error_type",
    "lexical_overlap",
}

print("Configuration locked.")
print(f"FSC root:          {FSC_ROOT}")
print(f"ASR feature count:    {len(ASR_FEATURES)}")
print(f"Intent feature count: {len(INTENT_FEATURES)}")
print(f"RUN_ASR_TRAINING:    {RUN_ASR_TRAINING}")
print(f"RUN_INTENT_TRAINING: {RUN_INTENT_TRAINING}")


Configuration locked.
FSC root:          C:\Users\ACER\OneDrive\Desktop\VoxIntel\data\raw\fsc\fluent_speech_commands_dataset
ASR feature count:    9
Intent feature count: 3
RUN_ASR_TRAINING:    True
RUN_INTENT_TRAINING: True


## 04 — Dataset download / structure verification

FSC is ~30k audio files — this notebook does **not** download it automatically. It only
verifies the expected structure and fails loudly with instructions if the data is
missing.

Official dataset page:
[Fluent Speech Commands](https://fluent.ai/fluent-speech-commands-a-dataset-for-spoken-language-understanding-research/)

Download and extract under `data/raw/fsc/` before running the cell below.

In [4]:
FSC_SETUP_INSTRUCTIONS = f"""
FSC dataset not found.

Download:
  https://fluent.ai/fluent-speech-commands-a-dataset-for-spoken-language-understanding-research/

Expected structure after extraction:
  {FSC_ROOT}/
      data/
          train_data.csv
          valid_data.csv
          test_data.csv
      wavs/
          speakers/
"""

missing = [
    p
    for p in [
        FSC_ROOT,
        FSC_TRAIN_CSV,
        FSC_VALID_CSV,
        FSC_TEST_CSV,
        FSC_AUDIO_DIR,
    ]
    if not p.exists()
]

if missing:
    print(FSC_SETUP_INSTRUCTIONS)
    print("Missing paths:")
    for p in missing:
        print(f"  - {p}")

    raise FileNotFoundError(
        "FSC dataset is not present at the expected location."
    )

print("FSC dataset structure verified.")
print(f"  train: {FSC_TRAIN_CSV}")
print(f"  valid: {FSC_VALID_CSV}")
print(f"  test:  {FSC_TEST_CSV}")
print(f"  audio: {FSC_AUDIO_DIR}")

FSC dataset structure verified.
  train: C:\Users\ACER\OneDrive\Desktop\VoxIntel\data\raw\fsc\fluent_speech_commands_dataset\data\train_data.csv
  valid: C:\Users\ACER\OneDrive\Desktop\VoxIntel\data\raw\fsc\fluent_speech_commands_dataset\data\valid_data.csv
  test:  C:\Users\ACER\OneDrive\Desktop\VoxIntel\data\raw\fsc\fluent_speech_commands_dataset\data\test_data.csv
  audio: C:\Users\ACER\OneDrive\Desktop\VoxIntel\data\raw\fsc\fluent_speech_commands_dataset\wavs


## 05 — FSC dataset audit

Reports split sizes against the official published numbers and runs integrity checks
before any modeling happens — missing files, duplicates, and (critically) **speaker
overlap across splits**, since FSC's value here depends on its splits actually being
speaker-independent rather than just documented as such.

In [5]:
train_df = pd.read_csv(FSC_TRAIN_CSV)
valid_df = pd.read_csv(FSC_VALID_CSV)
test_df = pd.read_csv(FSC_TEST_CSV)

EXPECTED_COUNTS = {"train": 23132, "validation": 3118, "test": 3793}
actual_counts = {"train": len(train_df), "validation": len(valid_df), "test": len(test_df)}

print("Split sizes (actual vs. officially published):")
for split, expected in EXPECTED_COUNTS.items():
    actual = actual_counts[split]
    flag = "OK" if actual == expected else "MISMATCH — proceeding with actual counts"
    print(f"  {split:<12} actual={actual:<7} expected={expected:<7} [{flag}]")

# Integrity checks -------------------------------------------------------
audit_rows = []
for name, df in [("train", train_df), ("validation", valid_df), ("test", test_df)]:
    n_missing_audio = df["path"].apply(
        lambda p: not (FSC_ROOT / p).exists()
    ).sum() if "path" in df.columns else None
    n_dup_ids = df.duplicated(subset=[df.columns[0]]).sum()
    n_dup_transcripts = df["transcription"].duplicated().sum() if "transcription" in df.columns else None
    audit_rows.append({
        "split": name,
        "n_rows": len(df),
        "n_missing_audio": n_missing_audio,
        "n_duplicate_ids": n_dup_ids,
        "n_duplicate_transcripts": n_dup_transcripts,
        "n_speakers": df["speakerId"].nunique() if "speakerId" in df.columns else None,
        "n_intents": (df["action"] + "|" + df["object"] + "|" + df["location"]).nunique()
            if set(["action", "object", "location"]).issubset(df.columns) else None,
    })

audit_df = pd.DataFrame(audit_rows)
print()
print(audit_df.to_string(index=False))

# Critical check: splits must be speaker-disjoint -------------------------
if "speakerId" in train_df.columns:
    train_speakers = set(train_df["speakerId"])
    valid_speakers = set(valid_df["speakerId"])
    test_speakers = set(test_df["speakerId"])

    assert train_speakers.isdisjoint(valid_speakers), "Speaker leakage: train/validation overlap"
    assert train_speakers.isdisjoint(test_speakers), "Speaker leakage: train/test overlap"
    assert valid_speakers.isdisjoint(test_speakers), "Speaker leakage: validation/test overlap"

    print()
    print(f"Speaker-independence check PASSED "
          f"(train={len(train_speakers)}, valid={len(valid_speakers)}, test={len(test_speakers)} speakers)")

audit_df.to_csv(FSC_AUDIT_PATH, index=False)
print(f"\nSaved: {FSC_AUDIT_PATH}")

Split sizes (actual vs. officially published):
  train        actual=23132   expected=23132   [OK]
  validation   actual=3118    expected=3118    [OK]
  test         actual=3793    expected=3793    [OK]

     split  n_rows  n_missing_audio  n_duplicate_ids  n_duplicate_transcripts  n_speakers  n_intents
     train   23132                0                0                    22884          77         31
validation    3118                0                0                     2870          10         31
      test    3793                0                0                     3545          10         31

Speaker-independence check PASSED (train=77, valid=10, test=10 speakers)

Saved: C:\Users\ACER\OneDrive\Desktop\VoxIntel\reports\fsc_dataset_audit.csv


## 06 — Label mapping

Intent labels are built **from training data only** — never from validation or test — to
avoid quietly leaking label-space information from held-out splits. FSC's `action` /
`object` / `location` columns are combined into a single intent label, matching the
single-label classification setup used for SLURP intent in Notebook 07.

In [6]:
def fsc_intent_label(df: pd.DataFrame) -> pd.Series:
    return df["action"].astype(str) + "|" + df["object"].astype(str) + "|" + df["location"].astype(str)

train_df["intent_label"] = fsc_intent_label(train_df)
valid_df["intent_label"] = fsc_intent_label(valid_df)
test_df["intent_label"] = fsc_intent_label(test_df)

# Label space is derived from TRAIN ONLY.
unique_train_intents = sorted(train_df["intent_label"].unique())
intent_to_id = {intent: i for i, intent in enumerate(unique_train_intents)}
id_to_intent = {i: intent for intent, i in intent_to_id.items()}

n_labels = len(intent_to_id)
print(f"Number of labels (from training data): {n_labels}")
assert n_labels == 31, f"Expected 31 FSC intents, found {n_labels}"

# Any validation/test intent not seen in training is dropped from modeling
# and reported explicitly rather than silently coerced.
for name, df in [("validation", valid_df), ("test", test_df)]:
    unseen = set(df["intent_label"]) - set(intent_to_id)
    if unseen:
        print(f"WARNING: {len(unseen)} unseen intent label(s) in {name}, dropping those rows: {unseen}")
        df.drop(df[df["intent_label"].isin(unseen)].index, inplace=True)

train_df["intent_id"] = train_df["intent_label"].map(intent_to_id)
valid_df["intent_id"] = valid_df["intent_label"].map(intent_to_id)
test_df["intent_id"] = test_df["intent_label"].map(intent_to_id)

print("Label mapping built from training data only.")

Number of labels (from training data): 31
Label mapping built from training data only.


## 07 — Train FSC ASR Transformer

Fine-tunes a **fresh** `wav2vec2-base-960h` checkpoint on FSC training audio. This is a
new checkpoint — the SLURP fine-tuned ASR model is not reused, since the whole point of
this notebook is an independently trained model on an independent dataset.

    FSC train
        ↓
    Wav2Vec2ForCTC
        ↓
    FSC ASR checkpoint  →  models/wav2vec2_fsc/

`finetune_wav2vec2_ctc(...)` is fully implemented below (frozen feature encoder, dynamic
padding, HF `Trainer`). It only actually runs when `RUN_ASR_TRAINING = True` (Section 03) —
otherwise this cell reports the checkpoint target and whether one already exists, so opening
the notebook never accidentally starts a multi-hour training job.


In [ ]:
def raw_ctc_diagnostic(model, processor, train_df, audio_root, n_batches=5, batch_size=8, use_fp16=False):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = model.to(device)
    model.freeze_feature_encoder()
    model.config.ctc_zero_infinity = False
    model.train()

    rows = train_df.sample(n_batches * batch_size, random_state=0).reset_index(drop=True)

    for b in range(n_batches):
        batch_rows = rows.iloc[b*batch_size:(b+1)*batch_size]
        audios, labels_list = [], []
        for _, row in batch_rows.iterrows():
            path = resolve_fsc_audio_path(audio_root, row["path"])
            audio = load_audio(path)
            if isinstance(audio, (tuple, list)):
                audio = audio[0]
            audio = np.asarray(audio, dtype=np.float32).squeeze()
            audios.append(audio)
            ids = processor.tokenizer(str(row["transcription"]).upper()).input_ids
            labels_list.append(ids)

        max_a = max(len(a) for a in audios)
        input_values = torch.zeros(len(audios), max_a)
        attention_mask = torch.zeros(len(audios), max_a, dtype=torch.long)
        for i, a in enumerate(audios):
            input_values[i, :len(a)] = torch.tensor(a)
            attention_mask[i, :len(a)] = 1

        max_l = max(len(l) for l in labels_list)
        labels = torch.full((len(labels_list), max_l), -100, dtype=torch.long)
        for i, l in enumerate(labels_list):
            labels[i, :len(l)] = torch.tensor(l)

        input_values, attention_mask, labels = input_values.to(device), attention_mask.to(device), labels.to(device)

        with torch.autocast(device_type="cuda", enabled=use_fp16):
            out = model(input_values=input_values, attention_mask=attention_mask, labels=labels)

        print(f"batch {b}: loss={out.loss.item():.4f} isnan={torch.isnan(out.loss).item()} isinf={torch.isinf(out.loss).item()}")

In [9]:
   processor = AutoProcessor.from_pretrained(BASE_ASR_MODEL)
   asr_model = Wav2Vec2ForCTC.from_pretrained(BASE_ASR_MODEL)

   with torch.no_grad():
       asr_model.wav2vec2.masked_spec_embed.uniform_()

   bad = [n for n, p in asr_model.named_parameters() if torch.isnan(p).any() or torch.isinf(p).any()]
   grad_ok = torch.is_grad_enabled()
   print("bad params:", bad)
   print("grad enabled globally:", grad_ok)
   assert not bad and grad_ok, "not safe to train yet"

Loading weights: 100%|██████████| 212/212 [00:00<00:00, 306.85it/s, Materializing param=wav2vec2.feature_projection.projection.weight]                         
Wav2Vec2ForCTC LOAD REPORT from: facebook/wav2vec2-base-960h
Key                        | Status  | 
---------------------------+---------+-
wav2vec2.masked_spec_embed | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


bad params: []
grad enabled globally: True


In [10]:
def resolve_fsc_audio_path(audio_root: Path, relative_path) -> Path:
    relative_path = Path(str(relative_path))

    if relative_path.parts and relative_path.parts[0].lower() == "wavs":
        relative_path = Path(*relative_path.parts[1:])

    resolved = audio_root / relative_path

    if not resolved.exists():
        raise FileNotFoundError(
            f"Audio file not found.\n"
            f"  CSV path:      {relative_path}\n"
            f"  Audio root:    {audio_root}\n"
            f"  Resolved path: {resolved}"
        )

    return resolved


def finetune_wav2vec2_ctc(
    model,
    processor,
    train_df,
    valid_df,
    audio_root: Path,
    output_dir: Path,
    num_epochs: int = 10,
    batch_size: int = 8,
    learning_rate: float = 3e-4,
):

    from dataclasses import dataclass
    from typing import Any, Dict, List
    from torch.utils.data import Dataset as TorchDataset

    class FSCCTCDataset(TorchDataset):

        def __init__(
            self,
            df: pd.DataFrame,
            audio_root: Path,
            processor,
        ):
            self.df = df.reset_index(drop=True)
            self.audio_root = audio_root
            self.processor = processor

        def __len__(self):
            return len(self.df)

        def __getitem__(self, idx: int) -> Dict[str, Any]:

            row = self.df.iloc[idx]

            audio_path = resolve_fsc_audio_path(
                self.audio_root,
                row["path"],
            )

            audio = load_audio(audio_path)

            if isinstance(audio, (tuple, list)):
                audio = audio[0]

            if torch.is_tensor(audio):
                audio = audio.detach().cpu().numpy()

            audio = np.asarray(
                audio,
                dtype=np.float32,
            ).squeeze()

            if audio.ndim != 1:
                raise ValueError(
                    f"Expected 1-D waveform, got shape "
                    f"{audio.shape} for {audio_path}"
                )

            processed_audio = self.processor(
                audio,
                sampling_rate=16000,
            )

            input_values = np.asarray(
                processed_audio.input_values[0],
                dtype=np.float32,
            )

            label_ids = self.processor.tokenizer(
    str(row["transcription"]).upper()
).input_ids

            label_ids = np.asarray(
                label_ids,
                dtype=np.int64,
            )

            return {
                "input_values": input_values,
                "labels": label_ids,
            }


    @dataclass
    class DataCollatorCTCWithPadding:

        processor: Any

        def __call__(
            self,
            features: List[Dict[str, Any]],
        ) -> Dict[str, torch.Tensor]:

            audio_tensors = [
                torch.as_tensor(
                    feature["input_values"],
                    dtype=torch.float32,
                )
                for feature in features
            ]

            label_tensors = [
                torch.as_tensor(
                    feature["labels"],
                    dtype=torch.long,
                )
                for feature in features
            ]

            max_audio_length = max(
                tensor.size(0)
                for tensor in audio_tensors
            )

            batch_size = len(audio_tensors)

            input_values = torch.zeros(
                batch_size,
                max_audio_length,
                dtype=torch.float32,
            )

            attention_mask = torch.zeros(
                batch_size,
                max_audio_length,
                dtype=torch.long,
            )

            for i, audio_tensor in enumerate(audio_tensors):

                length = audio_tensor.size(0)

                input_values[i, :length] = audio_tensor
                attention_mask[i, :length] = 1

            max_label_length = max(
                tensor.size(0)
                for tensor in label_tensors
            )

            labels = torch.full(
                (batch_size, max_label_length),
                -100,
                dtype=torch.long,
            )

            for i, label_tensor in enumerate(label_tensors):

                length = label_tensor.size(0)

                labels[i, :length] = label_tensor

            return {
                "input_values": input_values,
                "attention_mask": attention_mask,
                "labels": labels,
            }


    train_dataset = FSCCTCDataset(
        train_df,
        audio_root,
        processor,
    )

    valid_dataset = FSCCTCDataset(
        valid_df,
        audio_root,
        processor,
    )

    data_collator = DataCollatorCTCWithPadding(
        processor=processor,
    )

    model.freeze_feature_encoder()
    model.config.ctc_zero_infinity = True

    training_args = TrainingArguments(
        output_dir=str(output_dir),

        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,

        eval_strategy="epoch",
        save_strategy="epoch",

        num_train_epochs=num_epochs,
        learning_rate=learning_rate,
        warmup_steps=500,

        fp16=torch.cuda.is_available(),

        save_total_limit=2,

        load_best_model_at_end=True,

        metric_for_best_model="loss",
        greater_is_better=False,

        logging_steps=50,

        report_to=[],

        seed=RANDOM_SEED,
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=valid_dataset,
        data_collator=data_collator,
        processing_class=processor.feature_extractor,
    )

    trainer.train()

    trainer.save_model(str(output_dir))
    processor.save_pretrained(str(output_dir))

    return trainer


if RUN_ASR_TRAINING:

    print("Checking FSC audio paths before training...")

    for split_name, df in [
        ("train", train_df),
        ("validation", valid_df),
    ]:

        for i in range(min(10, len(df))):
            resolve_fsc_audio_path(
                FSC_AUDIO_DIR,
                df.iloc[i]["path"],
            )

        print(f"  {split_name}: first 10 audio paths OK")

    print("FSC audio path check passed.")

    print("Testing dataset and collator...")

    test_dataset = None

    print("Starting Wav2Vec2 FSC fine-tuning...")

    finetune_wav2vec2_ctc(
        asr_model,
        processor,
        train_df,
        valid_df,
        audio_root=FSC_AUDIO_DIR,
        output_dir=FSC_ASR_MODEL_DIR,
    )

    print(
        f"FSC ASR checkpoint saved: "
        f"{FSC_ASR_MODEL_DIR}"
    )

else:

    print(
        f"FSC ASR checkpoint target: "
        f"{FSC_ASR_MODEL_DIR}"
    )

    print(
        f"Checkpoint present: "
        f"{_dir_has_checkpoint(FSC_ASR_MODEL_DIR)}"
    )

    if not _dir_has_checkpoint(FSC_ASR_MODEL_DIR):

        print(
            "RUN_ASR_TRAINING is False — skipping training. "
            "Set RUN_ASR_TRAINING = True in Section 03 to fine-tune."
        )

Checking FSC audio paths before training...
  train: first 10 audio paths OK
  validation: first 10 audio paths OK
FSC audio path check passed.
Testing dataset and collator...
Starting Wav2Vec2 FSC fine-tuning...


Epoch,Training Loss,Validation Loss
1,61.867866,43.954926
2,50.543550,38.519749
3,41.039644,38.330013
4,44.709844,28.338774
5,24.356731,33.710281
6,21.946904,28.123688
7,17.692667,22.638805
8,23.764578,21.287842
9,18.726958,21.265331
10,16.008525,20.275986


Writing model shards: 100%|██████████| 1/1 [00:04<00:00,  4.27s/it]

FSC ASR checkpoint saved: C:\Users\ACER\OneDrive\Desktop\VoxIntel\models\wav2vec2_fsc


## 08 — FSC ASR evaluation

Establishes the upstream ASR baseline (WER/CER on validation and test). This is reported
for context only — **WER/CER never enter the risk feature matrix** (Section 15 enforces
this).

In [33]:
import re


def normalize_asr_text(text: str) -> str:
    """
    Normalize ASR/reference text before WER/CER evaluation.

    Evaluation-only normalization:
      - converts case to uppercase
      - normalizes repeated whitespace

    IMPORTANT:
    These normalized reference strings are used ONLY for
    upstream ASR evaluation. They are never passed to the
    VoxIntel-R feature matrix.
    """
    text = str(text).strip().upper()
    text = re.sub(r"\s+", " ", text)
    return text


def _levenshtein(ref: list, hyp: list) -> int:
    """Standard Levenshtein edit distance."""
    n, m = len(ref), len(hyp)

    dp = list(range(m + 1))

    for i in range(1, n + 1):

        prev = dp[0]
        dp[0] = i

        for j in range(1, m + 1):

            cur = dp[j]

            cost = (
                0
                if ref[i - 1] == hyp[j - 1]
                else 1
            )

            dp[j] = min(
                dp[j] + 1,        # deletion
                dp[j - 1] + 1,    # insertion
                prev + cost,      # substitution
            )

            prev = cur

    return dp[m]


def compute_wer_cer(
    predictions: list,
    references: list,
) -> dict:
    """
    Compute normalized WER/CER.

    IMPORTANT:
    We intentionally do NOT call jiwer here.

    The previous Section 08 implementation produced an
    impossible ~99% WER / ~82% CER despite exact transcript
    matches such as:

        REF: Turn on the lights
        ASR: TURN ON THE LIGHTS

    This implementation performs the validated normalization
    explicitly before edit-distance computation.
    """

    assert len(predictions) == len(references), (
        f"Prediction/reference length mismatch: "
        f"{len(predictions)} vs {len(references)}"
    )

    normalized_predictions = [
        normalize_asr_text(pred)
        for pred in predictions
    ]

    normalized_references = [
        normalize_asr_text(ref)
        for ref in references
    ]
    total_word_edits = 0
    total_reference_words = 0

    total_char_edits = 0
    total_reference_chars = 0

    for ref, hyp in zip(
        normalized_references,
        normalized_predictions,
    ):

        ref_words = ref.split()
        hyp_words = hyp.split()

        total_word_edits += _levenshtein(
            ref_words,
            hyp_words,
        )

        total_reference_words += max(
            len(ref_words),
            1,
        )
        
        # Ignore spaces for CER.
        # This makes CER measure character substitutions,
        # insertions and deletions rather than whitespace
        # formatting differences.

        ref_chars = list(
            ref.replace(" ", "")
        )

        hyp_chars = list(
            hyp.replace(" ", "")
        )

        total_char_edits += _levenshtein(
            ref_chars,
            hyp_chars,
        )

        total_reference_chars += max(
            len(ref_chars),
            1,
        )

    wer = (
        total_word_edits
        / total_reference_words
    )

    cer = (
        total_char_edits
        / total_reference_chars
    )

    return {
        "wer": float(wer),
        "cer": float(cer),
    }

def transcribe_dataset(
    asr_model,
    processor,
    df: pd.DataFrame,
    audio_root: Path,
) -> pd.DataFrame:

    records = []

    device = (
        "cuda"
        if torch.cuda.is_available()
        else "cpu"
    )

    asr_model = asr_model.to(device)
    asr_model.eval()

    for _, row in df.iterrows():

        audio_path = resolve_fsc_audio_path(
            audio_root,
            row["path"],
        )

        audio = load_audio(audio_path)

        if isinstance(audio, (tuple, list)):
            audio = audio[0]

        audio = np.asarray(
            audio,
            dtype=np.float32,
        ).squeeze()

        if audio.ndim != 1:
            raise ValueError(
                f"Expected 1-D waveform, got "
                f"{audio.shape} for {audio_path}"
            )

        inputs = processor(
            audio,
            sampling_rate=16000,
            return_tensors="pt",
        )

        inputs = {
            key: value.to(device)
            for key, value in inputs.items()
        }

        with torch.no_grad():

            logits = (
                asr_model(**inputs)
                .logits
                .squeeze(0)
                .cpu()
            )

        pred_ids = logits.argmax(
            dim=-1
        )

        hypothesis = (
            processor.batch_decode(
                pred_ids.unsqueeze(0)
            )[0]
        )

        records.append(
            {
                "utterance_id": row.get(
                    "path",
                    None,
                ),

                "asr_hypothesis": hypothesis,

                "audio_duration": (
                    audio.shape[-1]
                    / 16000
                ),

                "_logits": logits,
            }
        )

    return pd.DataFrame(records)

if _dir_has_checkpoint(FSC_ASR_MODEL_DIR):

    print("=" * 80)
    print("LOADING FSC ASR CHECKPOINT")
    print("=" * 80)

    fsc_asr_model = (
        Wav2Vec2ForCTC.from_pretrained(
            FSC_ASR_MODEL_DIR
        )
    )

    fsc_processor = (
        AutoProcessor.from_pretrained(
            FSC_ASR_MODEL_DIR
        )
    )

    print(
        f"Checkpoint: {FSC_ASR_MODEL_DIR}"
    )

    print()
    print("Transcribing FSC validation split...")

    valid_asr_results = transcribe_dataset(
        fsc_asr_model,
        fsc_processor,
        valid_df,
        FSC_AUDIO_DIR,
    )

    print(
        f"Validation utterances: "
        f"{len(valid_asr_results):,}"
    )

    print()
    print("Transcribing FSC test split...")

    test_asr_results = transcribe_dataset(
        fsc_asr_model,
        fsc_processor,
        test_df,
        FSC_AUDIO_DIR,
    )

    print(
        f"Test utterances: "
        f"{len(test_asr_results):,}"
    )

    valid_wer_cer = compute_wer_cer(
        predictions=(
            valid_asr_results[
                "asr_hypothesis"
            ].tolist()
        ),
        references=(
            valid_df[
                "transcription"
            ].tolist()
        ),
    )

    test_wer_cer = compute_wer_cer(
        predictions=(
            test_asr_results[
                "asr_hypothesis"
            ].tolist()
        ),
        references=(
            test_df[
                "transcription"
            ].tolist()
        ),
    )

    prediction_df = pd.concat(
        [

            valid_df[
                ["path", "transcription"]
            ].assign(
                split="validation",
                asr_hypothesis=(
                    valid_asr_results[
                        "asr_hypothesis"
                    ].values
                ),
            ),

            test_df[
                ["path", "transcription"]
            ].assign(
                split="test",
                asr_hypothesis=(
                    test_asr_results[
                        "asr_hypothesis"
                    ].values
                ),
            ),

        ],
        ignore_index=True,
    )

    prediction_df.to_csv(
        FSC_ASR_PRED_PATH,
        index=False,
    )

    print()
    print("=" * 80)
    print("FSC ASR — CORRECTED WER/CER")
    print("=" * 80)

    print(
        f"Validation WER: "
        f"{valid_wer_cer['wer']:.6f}"
    )

    print(
        f"Validation CER: "
        f"{valid_wer_cer['cer']:.6f}"
    )

    print(
        f"Test WER:       "
        f"{test_wer_cer['wer']:.6f}"
    )

    print(
        f"Test CER:       "
        f"{test_wer_cer['cer']:.6f}"
    )

    print()
    print(
        f"Saved: {FSC_ASR_PRED_PATH}"
    )

    assert len(valid_asr_results) == len(
        valid_df
    )

    assert len(test_asr_results) == len(
        test_df
    )

    assert 0.0 <= valid_wer_cer["wer"] <= 1.0
    assert 0.0 <= valid_wer_cer["cer"] <= 1.0

    assert 0.0 <= test_wer_cer["wer"] <= 1.0
    assert 0.0 <= test_wer_cer["cer"] <= 1.0

    print()
    print(
        "ASR evaluation integrity checks: PASS"
    )

else:

    fsc_asr_model = None
    fsc_processor = None

    valid_asr_results = None
    test_asr_results = None

    print(
        "FSC ASR checkpoint not found — "
        "run Section 07 first "
        "(set RUN_ASR_TRAINING = True)."
    )


print()
print(
    "WER/CER is an upstream sanity check only — "
    "never a risk-model input."
)

LOADING FSC ASR CHECKPOINT


Loading weights: 100%|██████████| 213/213 [00:00<00:00, 316.24it/s, Materializing param=wav2vec2.masked_spec_embed]                                            


Checkpoint: C:\Users\ACER\OneDrive\Desktop\VoxIntel\models\wav2vec2_fsc

Transcribing FSC validation split...
Validation utterances: 3,118

Transcribing FSC test split...
Test utterances: 3,793

FSC ASR — CORRECTED WER/CER
Validation WER: 0.032715
Validation CER: 0.027346
Test WER:       0.017975
Test CER:       0.018359

Saved: C:\Users\ACER\OneDrive\Desktop\VoxIntel\reports\fsc_asr_predictions.csv

ASR evaluation integrity checks: PASS

WER/CER is an upstream sanity check only — never a risk-model input.


## 09 — Train FSC intent Transformer

Fine-tunes a fresh `distilbert-base-uncased` as a 31-class intent classifier on FSC
**reference transcripts** — same architecture family as Notebook 07's SLURP intent model,
new checkpoint, new dataset.

    FSC reference transcripts
            ↓
        DistilBERT
            ↓
    31-class intent classifier  →  models/distilbert_fsc_intent/

`finetune_intent_classifier(...)` is fully implemented below (HF `Trainer`, dynamic padding
via `DataCollatorWithPadding`). It only actually runs when `RUN_INTENT_TRAINING = True`
(Section 03) for the same reason as Section 07.


In [13]:
tokenizer = AutoTokenizer.from_pretrained(BASE_INTENT_MODEL)

intent_model = AutoModelForSequenceClassification.from_pretrained(
    BASE_INTENT_MODEL,
    num_labels=n_labels,
)


def tokenize_intent_batch(texts: list[str]):
    return tokenizer(
        texts,
        padding=True,
        truncation=True,
        return_tensors="pt",
    )


def finetune_intent_classifier(
    model,
    tokenizer,
    train_df,
    valid_df,
    output_dir: Path,
    num_epochs: int = 5,
    batch_size: int = 16,
    learning_rate: float = 2e-5,
    max_length: int = 64,
):
    """
    Fine-tune DistilBERT for FSC intent classification on ground-truth
    transcripts (`transcription` column), analogous to Notebook 07.
    """
    from torch.utils.data import Dataset as TorchDataset

    class FSCIntentDataset(TorchDataset):
        def __init__(self, df: pd.DataFrame):
            self.texts = df["transcription"].astype(str).tolist()
            self.labels = df["intent_id"].astype(int).tolist()

        def __len__(self) -> int:
            return len(self.texts)

        def __getitem__(self, idx: int) -> dict:
            enc = tokenizer(
                self.texts[idx],
                truncation=True,
                max_length=max_length,
            )
            enc["labels"] = self.labels[idx]
            return enc

    train_dataset = FSCIntentDataset(train_df)
    valid_dataset = FSCIntentDataset(valid_df)
    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        preds = logits.argmax(axis=-1)
        return {
            "accuracy": accuracy_score(labels, preds),
            "macro_f1": f1_score(labels, preds, average="macro"),
        }

    training_args = TrainingArguments(
        output_dir=str(output_dir),
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        eval_strategy="epoch",
        save_strategy="epoch",
        num_train_epochs=num_epochs,
        learning_rate=learning_rate,
        weight_decay=0.01,
        load_best_model_at_end=True,
        metric_for_best_model="accuracy",
        greater_is_better=True,
        logging_steps=50,
        report_to=[],
        seed=RANDOM_SEED,
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=valid_dataset,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )

    trainer.train()
    trainer.save_model(str(output_dir))
    tokenizer.save_pretrained(output_dir)
    return trainer


if RUN_INTENT_TRAINING:
    finetune_intent_classifier(
        intent_model,
        tokenizer,
        train_df,
        valid_df,
        output_dir=FSC_INTENT_MODEL_DIR,
    )
    print(f"FSC intent checkpoint saved: {FSC_INTENT_MODEL_DIR}")
else:
    print(f"FSC intent checkpoint target: {FSC_INTENT_MODEL_DIR}")
    print(f"Checkpoint present: {_dir_has_checkpoint(FSC_INTENT_MODEL_DIR)}")
    if not _dir_has_checkpoint(FSC_INTENT_MODEL_DIR):
        print("RUN_INTENT_TRAINING is False — skipping training. "
              "Set RUN_INTENT_TRAINING = True in Section 03 to fine-tune.")


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 351.66it/s, Materializing param=distilbert.transformer.layer.5.sa_layer_norm.weight]   
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,0.005256,0.002622,1.000000,1.000000
2,0.001309,0.000647,1.000000,1.000000
3,0.000579,0.000277,1.000000,1.000000
4,0.000334,0.000157,1.000000,1.000000
5,0.000262,0.000122,1.000000,1.000000


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.81it/s]
There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.34it/s]

FSC intent checkpoint saved: C:\Users\ACER\OneDrive\Desktop\VoxIntel\models\distilbert_fsc_intent


## 10 — Clean-text intent upper bound

The FSC analogue of Notebook 07: intent performance on the **ground-truth transcript**,
establishing the ceiling before ASR errors are introduced.

In [14]:
def top_k_accuracy(logits: torch.Tensor, labels: torch.Tensor, k: int = 3) -> float:
    topk = logits.topk(k, dim=-1).indices
    correct = (topk == labels.unsqueeze(-1)).any(dim=-1)
    return correct.float().mean().item()


def run_intent_inference(
    model,
    tokenizer,
    texts,
    labels,
    batch_size: int = 32,
    max_length: int = 64,
    device: str | None = None,
):
    """
    Batched inference over the intent classifier. This is the function
    Sections 10-11 call but that was previously never defined.

    Returns:
        logits: torch.Tensor of shape (N, num_labels)
        labels: np.ndarray of shape (N,) — passed through for convenient
            pairing with `logits` by the caller.
    """
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    model.eval()

    texts = list(texts)
    all_logits = []

    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            batch_texts = texts[i : i + batch_size]
            enc = tokenizer(
                batch_texts,
                padding=True,
                truncation=True,
                max_length=max_length,
                return_tensors="pt",
            ).to(device)
            out = model(**enc)
            all_logits.append(out.logits.cpu())

    logits = torch.cat(all_logits, dim=0)
    return logits, np.asarray(labels)


if _dir_has_checkpoint(FSC_INTENT_MODEL_DIR):
    fsc_intent_model = AutoModelForSequenceClassification.from_pretrained(FSC_INTENT_MODEL_DIR)
    fsc_tokenizer = AutoTokenizer.from_pretrained(FSC_INTENT_MODEL_DIR)

    clean_logits, clean_labels = run_intent_inference(
        fsc_intent_model, fsc_tokenizer, test_df["transcription"], test_df["intent_id"]
    )
    clean_preds = clean_logits.argmax(dim=-1).numpy()

    clean_text_metrics = {
        "accuracy": accuracy_score(clean_labels, clean_preds),
        "macro_f1": f1_score(clean_labels, clean_preds, average="macro"),
        "weighted_f1": f1_score(clean_labels, clean_preds, average="weighted"),
        "top3_accuracy": top_k_accuracy(clean_logits, torch.tensor(clean_labels), k=3),
    }
    print("Clean-text intent upper bound (FSC test):")
    for k, v in clean_text_metrics.items():
        print(f"  {k}: {v:.4f}")
else:
    fsc_intent_model = None
    fsc_tokenizer = None
    clean_text_metrics = None
    print("FSC intent checkpoint not found — run Section 09 first "
          "(set RUN_INTENT_TRAINING = True).")

print("Clean-text upper bound establishes the ceiling for the ASR-propagated numbers in Section 11.")


Loading weights: 100%|██████████| 104/104 [00:00<00:00, 357.62it/s, Materializing param=pre_classifier.weight]                                  


Clean-text intent upper bound (FSC test):
  accuracy: 1.0000
  macro_f1: 1.0000
  weighted_f1: 1.0000
  top3_accuracy: 1.0000
Clean-text upper bound establishes the ceiling for the ASR-propagated numbers in Section 11.


## 11 — ASR → intent propagation

Reproduces Notebook 08's logic on FSC: run the FSC ASR hypothesis through the FSC intent
model, and compare against the clean-transcript ceiling from Section 10.

    audio
     ↓
    FSC Wav2Vec2  →  ASR hypothesis
     ↓
    FSC DistilBERT  →  intent prediction

In [15]:
def propagate_intent(asr_results: pd.DataFrame, source_df: pd.DataFrame) -> pd.DataFrame:
    """
    Runs the FSC intent model over an already-computed ASR hypothesis
    (`asr_results["asr_hypothesis"]`, produced in Section 08) and attaches
    ground-truth / predicted intent ids plus the raw intent logits needed
    for uncertainty extraction in Section 14.

    Note: this reuses `valid_asr_results` / `test_asr_results` from Section
    08 rather than re-transcribing, so audio only passes through the ASR
    model once.
    """
    logits, gt_ids = run_intent_inference(
        fsc_intent_model,
        fsc_tokenizer,
        asr_results["asr_hypothesis"],
        source_df["intent_id"],
    )
    out = asr_results.reset_index(drop=True).copy()
    out["ground_truth_intent_id"] = gt_ids
    out["predicted_intent_id"] = logits.argmax(dim=-1).numpy()
    out["_intent_logits"] = list(logits)
    return out


# ------------------------------------------------------------
# ASR → intent propagation, run on BOTH validation and test.
# Validation propagated features are needed to fit the Section 16 risk
# models; test propagated features are needed for the frozen evaluation.
# ------------------------------------------------------------
if (
    fsc_asr_model is not None
    and fsc_intent_model is not None
    and valid_asr_results is not None
    and test_asr_results is not None
):
    valid_propagated = propagate_intent(valid_asr_results, valid_df)
    test_propagated = propagate_intent(test_asr_results, test_df)

    propagated_metrics = {
        "accuracy": accuracy_score(
            test_propagated["ground_truth_intent_id"],
            test_propagated["predicted_intent_id"],
        ),
        "macro_f1": f1_score(
            test_propagated["ground_truth_intent_id"],
            test_propagated["predicted_intent_id"],
            average="macro",
        ),
    }

    print("Ground truth vs. ASR-propagated intent accuracy on FSC test:")
    if clean_text_metrics is not None:
        print(f"  clean:          {clean_text_metrics['accuracy']:.4f}")
    print(f"  ASR-propagated: {propagated_metrics['accuracy']:.4f}")
    if clean_text_metrics is not None:
        print(
            f"  recovery gap:   "
            f"{clean_text_metrics['accuracy'] - propagated_metrics['accuracy']:.4f}"
        )

    pd.concat([
        test_df[["path", "intent_label"]].assign(
            predicted_intent_id=test_propagated["predicted_intent_id"].values,
        ),
    ]).to_csv(FSC_INTENT_PRED_PATH, index=False)
    print(f"Saved: {FSC_INTENT_PRED_PATH}")
else:
    valid_propagated = None
    test_propagated = None
    print(
        "Propagation requires trained FSC ASR + intent checkpoints "
        "(Sections 07, 09) and their Section 08/10 outputs."
    )


Ground truth vs. ASR-propagated intent accuracy on FSC test:
  clean:          1.0000
  ASR-propagated: 0.9939
  recovery gap:   0.0061
Saved: C:\Users\ACER\OneDrive\Desktop\VoxIntel\reports\fsc_intent_predictions.csv


## 12 — Construct the failure target

The supervised target, exactly as defined for SLURP in Notebook 08. The ground-truth
intent is allowed to construct the **target** — it is never allowed into the inference-time
feature matrix (enforced in Section 15).

In [16]:
def build_failure_target(df: pd.DataFrame, predicted_col: str, ground_truth_col: str) -> pd.DataFrame:
    df = df.copy()
    df["intent_failed"] = (df[predicted_col] != df[ground_truth_col]).astype(int)
    return df


if valid_propagated is not None and test_propagated is not None:
    valid_risk_df = build_failure_target(
        valid_propagated, predicted_col="predicted_intent_id", ground_truth_col="ground_truth_intent_id"
    )
    test_risk_df = build_failure_target(
        test_propagated, predicted_col="predicted_intent_id", ground_truth_col="ground_truth_intent_id"
    )

    print(f"Validation failure prevalence: {valid_risk_df['intent_failed'].mean():.4f}")
    print(f"Test failure prevalence:       {test_risk_df['intent_failed'].mean():.4f}")
else:
    valid_risk_df = None
    test_risk_df = None
    print("Failure target requires Section 11's propagated results.")

print("Target definition fixed: intent_failed = (predicted_intent != ground_truth_intent).")


Validation failure prevalence: 0.0298
Test failure prevalence:       0.0061
Target definition fixed: intent_failed = (predicted_intent != ground_truth_intent).


## 13 — Extract ASR-native uncertainty

Same feature schema as Notebook 16, computed from CTC frame-level logits. Naming note
carried over from the SLURP notebook: these are **frame-level CTC uncertainty
statistics**, not true decoded-token confidence — the notebook names them accordingly
rather than overclaiming token-level confidence.

In [17]:
def extract_asr_uncertainty(
    logits: torch.Tensor,
    audio_duration: float,
) -> dict:
    """
    Frame-level CTC uncertainty statistics.

    `logits`: (T, vocab) raw CTC logits for one utterance.
    """
    probs = F.softmax(logits, dim=-1)

    frame_confidence = probs.max(dim=-1).values

    frame_entropy = -(
        probs * probs.clamp_min(1e-12).log()
    ).sum(dim=-1)

    return {
        "asr_mean_confidence": frame_confidence.mean().item(),
        "asr_min_confidence": frame_confidence.min().item(),
        "asr_std_confidence": frame_confidence.std(
            unbiased=False
        ).item(),
        "asr_median_confidence": frame_confidence.median().item(),
        "asr_mean_entropy": frame_entropy.mean().item(),
        "asr_max_entropy": frame_entropy.max().item(),
        "asr_std_entropy": frame_entropy.std(
            unbiased=False
        ).item(),
        "asr_num_frames": logits.shape[0],
        "audio_duration": audio_duration,
    }


def build_asr_feature_df(propagated_df: pd.DataFrame) -> pd.DataFrame:
    rows = [
        extract_asr_uncertainty(row["_logits"], row["audio_duration"])
        for _, row in propagated_df.iterrows()
    ]
    df = pd.DataFrame(rows)
    assert list(df.columns) == ASR_FEATURES
    return df


if valid_propagated is not None and test_propagated is not None:
    valid_asr_feature_df = build_asr_feature_df(valid_propagated)
    test_asr_feature_df = build_asr_feature_df(test_propagated)
else:
    valid_asr_feature_df = None
    test_asr_feature_df = None

print(
    f"ASR-native uncertainty schema "
    f"({len(ASR_FEATURES)} features): {ASR_FEATURES}"
)


ASR-native uncertainty schema (9 features): ['asr_mean_confidence', 'asr_min_confidence', 'asr_std_confidence', 'asr_median_confidence', 'asr_mean_entropy', 'asr_max_entropy', 'asr_std_entropy', 'asr_num_frames', 'audio_duration']


## 14 — Extract intent-native uncertainty

Same three features as Notebook 16, computed from the intent classifier's softmax output
over the ASR-propagated hypothesis.

In [19]:
def extract_intent_uncertainty(logits: torch.Tensor) -> dict:
    """
    Extract reference-free intent uncertainty features.

    logits:
        (num_labels,) raw classification logits for one utterance.

    IMPORTANT:
        The returned dictionary order intentionally matches
        INTENT_FEATURES exactly.
    """

    if not torch.is_tensor(logits):
        logits = torch.as_tensor(
            logits,
            dtype=torch.float32,
        )

    logits = logits.detach().float().cpu()

    probs = F.softmax(logits, dim=-1)

    sorted_probs, _ = torch.sort(
        probs,
        descending=True,
    )

    top1 = sorted_probs[0].item()
    top2 = sorted_probs[1].item()

    entropy = -(
        probs
        * probs.clamp_min(1e-12).log()
    ).sum().item()

    # Return in EXACTLY the same order as INTENT_FEATURES:
    #
    # [
    #     "intent_confidence",
    #     "intent_entropy",
    #     "intent_margin",
    # ]

    return {
        "intent_confidence": top1,
        "intent_entropy": entropy,
        "intent_margin": top1 - top2,
    }


def build_intent_feature_df(
    propagated_df: pd.DataFrame,
) -> pd.DataFrame:

    rows = [
        extract_intent_uncertainty(logit)
        for logit in propagated_df["_intent_logits"]
    ]

    df = pd.DataFrame(rows)

    # Explicitly enforce the locked VoxIntel-R schema.
    # This protects against dictionary-order changes and makes
    # the downstream A/B/C feature matrix deterministic.
    df = df.loc[:, INTENT_FEATURES]

    assert list(df.columns) == INTENT_FEATURES, (
        f"Intent feature schema mismatch.\n"
        f"Expected: {INTENT_FEATURES}\n"
        f"Actual:   {list(df.columns)}"
    )

    return df


if (
    valid_propagated is not None
    and test_propagated is not None
):

    valid_intent_feature_df = build_intent_feature_df(
        valid_propagated
    )

    test_intent_feature_df = build_intent_feature_df(
        test_propagated
    )

else:

    valid_intent_feature_df = None
    test_intent_feature_df = None


print(
    f"Intent-native uncertainty schema "
    f"({len(INTENT_FEATURES)} features): "
    f"{INTENT_FEATURES}"
)

print(
    f"Validation feature shape: "
    f"{valid_intent_feature_df.shape}"
    if valid_intent_feature_df is not None
    else "Validation features: None"
)

print(
    f"Test feature shape: "
    f"{test_intent_feature_df.shape}"
    if test_intent_feature_df is not None
    else "Test features: None"
)

print("STATUS: Intent feature schema PASS")

Intent-native uncertainty schema (3 features): ['intent_confidence', 'intent_entropy', 'intent_margin']
Validation feature shape: (3118, 3)
Test feature shape: (3793, 3)
STATUS: Intent feature schema PASS


## 15 — Hard leakage audit

The single most important cell in this notebook. Fails loudly — rather than silently
producing an invalid experiment — if any forbidden signal has made its way into the
feature matrix.

In [20]:
def leakage_audit(feature_columns: list[str]) -> None:
    overlap = set(feature_columns) & FORBIDDEN_FEATURES
    assert not overlap, f"LEAKAGE DETECTED — forbidden features present: {overlap}"


def assemble_full_feature_df(asr_feature_df, intent_feature_df, risk_df) -> pd.DataFrame:
    out = pd.concat(
        [asr_feature_df.reset_index(drop=True), intent_feature_df.reset_index(drop=True)],
        axis=1,
    )
    out["intent_failed"] = risk_df["intent_failed"].reset_index(drop=True)
    return out


if valid_asr_feature_df is not None and test_asr_feature_df is not None:
    valid_full = assemble_full_feature_df(
        valid_asr_feature_df, valid_intent_feature_df, valid_risk_df
    ).assign(split="validation")
    test_full = assemble_full_feature_df(
        test_asr_feature_df, test_intent_feature_df, test_risk_df
    ).assign(split="test")

    # `full_feature_df` is the single feature matrix the A/B/C harness
    # (Section 16) trains on `split == "validation"` and evaluates on
    # `split == "test"` — this is what was previously missing entirely.
    full_feature_df = pd.concat([valid_full, test_full], ignore_index=True)

    feature_columns = ASR_FEATURES + INTENT_FEATURES
    leakage_audit(feature_columns)

    full_feature_df.to_csv(FEATURE_PATH, index=False)

    print("REFERENCE-FREE AUDIT")
    print("-" * 20)
    print(f"Reference transcript in X: {'YES' if 'reference_transcript' in feature_columns else 'NO'}")
    print(f"Ground-truth intent in X:  {'YES' if 'ground_truth_intent' in feature_columns else 'NO'}")
    print(f"WER/CER in X:              {'YES' if ({'wer','cer'} & set(feature_columns)) else 'NO'}")
    print(f"Taxonomy in X:             {'YES' if 'taxonomy' in feature_columns else 'NO'}")
    print(f"ASR hypothesis available:  YES")
    print(f"ASR uncertainty available: YES")
    print(f"Intent uncertainty available: YES")
    print()
    print("STATUS: PASS")
    print(f"\nSaved: {FEATURE_PATH} ({len(full_feature_df)} rows: "
          f"{(full_feature_df['split'] == 'validation').sum()} validation, "
          f"{(full_feature_df['split'] == 'test').sum()} test)")
else:
    full_feature_df = None
    print(f"Forbidden feature set: {sorted(FORBIDDEN_FEATURES)}")
    print("Feature assembly skipped — requires Sections 12-14 outputs for both splits.")


REFERENCE-FREE AUDIT
--------------------
Reference transcript in X: NO
Ground-truth intent in X:  NO
WER/CER in X:              NO
Taxonomy in X:             NO
ASR hypothesis available:  YES
ASR uncertainty available: YES
Intent uncertainty available: YES

STATUS: PASS

Saved: C:\Users\ACER\OneDrive\Desktop\VoxIntel\reports\fsc_voxintel_r_features.csv (6911 rows: 3118 validation, 3793 test)


## 16 — A/B/C risk experiments

Same three experiment arms and the same two model families as Notebook 16 —
`LogisticRegression` and `RandomForestClassifier`. No new model families (no XGBoost,
LightGBM, or neural MLPs): the goal here is dataset/model-family validation, not a model
zoo.

- **A** — ASR-native uncertainty only
- **B** — Intent-native uncertainty only
- **C** — ASR + intent uncertainty combined

In [21]:
RISK_MODELS = {
    "logistic_regression": lambda: LogisticRegression(max_iter=1000, random_state=RANDOM_SEED),
    "random_forest": lambda: RandomForestClassifier(n_estimators=300, random_state=RANDOM_SEED),
}

EXPERIMENTS = {
    "A_asr_only": ASR_FEATURES,
    "B_intent_only": INTENT_FEATURES,
    "C_combined": ASR_FEATURES + INTENT_FEATURES,
}

def evaluate_risk_model(model, X_train, y_train, X_test, y_test) -> dict:
    model.fit(X_train, y_train)
    probs = model.predict_proba(X_test)[:, 1]
    preds = (probs >= 0.5).astype(int)
    return {
        "roc_auc": roc_auc_score(y_test, probs),
        "pr_auc": average_precision_score(y_test, probs),
        "brier": brier_score_loss(y_test, probs),
        "f1": f1_score(y_test, preds),
        "precision": precision_score(y_test, preds, zero_division=0),
        "recall": recall_score(y_test, preds, zero_division=0),
    }

def run_ab_c_experiments(feature_df: pd.DataFrame, target: pd.Series,
                          train_idx, test_idx) -> pd.DataFrame:
    rows = []
    for exp_name, feature_cols in EXPERIMENTS.items():
        X_train, X_test = feature_df.loc[train_idx, feature_cols], feature_df.loc[test_idx, feature_cols]
        y_train, y_test = target.loc[train_idx], target.loc[test_idx]
        for model_name, model_fn in RISK_MODELS.items():
            metrics = evaluate_risk_model(model_fn(), X_train, y_train, X_test, y_test)
            rows.append({"experiment": exp_name, "model": model_name, **metrics})
    return pd.DataFrame(rows)


if full_feature_df is not None:
    # Split policy (Section 30 of the design discussion): fit on FSC
    # validation, evaluate frozen on FSC test — never on the test set
    # the risk model was trained on.
    train_idx = full_feature_df.index[full_feature_df["split"] == "validation"]
    test_idx = full_feature_df.index[full_feature_df["split"] == "test"]

    risk_results_df = run_ab_c_experiments(
        full_feature_df, full_feature_df["intent_failed"],
        train_idx=train_idx, test_idx=test_idx,
    )
    risk_results_df.to_csv(RESULTS_PATH, index=False)
    print(risk_results_df.to_string(index=False))
    print(f"\nSaved: {RESULTS_PATH}")
else:
    risk_results_df = None
    train_idx = None
    test_idx = None
    print("A/B/C harness defined: fit on FSC validation, evaluate on FSC test "
          "(see Section 30 split policy) — run once full_feature_df is assembled (Section 15).")


   experiment               model  roc_auc   pr_auc    brier       f1  precision   recall
   A_asr_only logistic_regression 0.596483 0.228283 0.005429 0.296296       1.00 0.173913
   A_asr_only       random_forest 0.695139 0.172221 0.006214 0.222222       0.75 0.130435
B_intent_only logistic_regression 0.703304 0.259707 0.005706 0.160000       1.00 0.086957
B_intent_only       random_forest 0.735325 0.309460 0.006087 0.378378       0.50 0.304348
   C_combined logistic_regression 0.598951 0.270356 0.005191 0.296296       1.00 0.173913
   C_combined       random_forest 0.710610 0.269637 0.005632 0.296296       1.00 0.173913

Saved: C:\Users\ACER\OneDrive\Desktop\VoxIntel\reports\fsc_voxintel_r_model_comparison.csv


In [39]:
def paired_bootstrap_delta(
    y_true,
    probs_b,
    probs_c,
    metric_fn,
    n_bootstrap=5000,
    random_state=42,
):
    """
    Paired bootstrap comparison of model C against model B.

    Each bootstrap replicate resamples the SAME test examples
    for both models, preserving the paired structure.

    Returns:
        delta = metric(C) - metric(B)
        ci_lower
        ci_upper
        significant
    """

    y_true = np.asarray(y_true)
    probs_b = np.asarray(probs_b)
    probs_c = np.asarray(probs_c)

    assert len(y_true) == len(probs_b)
    assert len(y_true) == len(probs_c)

    rng = np.random.default_rng(
        random_state
    )

    n = len(y_true)

    # Observed difference
    observed_b = metric_fn(
        y_true,
        probs_b,
    )

    observed_c = metric_fn(
        y_true,
        probs_c,
    )

    observed_delta = (
        observed_c - observed_b
    )

    bootstrap_deltas = []


    for _ in range(n_bootstrap):

        indices = rng.integers(
            0,
            n,
            size=n,
        )

        y_boot = y_true[indices]
        b_boot = probs_b[indices]
        c_boot = probs_c[indices]

        try:

            score_b = metric_fn(
                y_boot,
                b_boot,
            )

            score_c = metric_fn(
                y_boot,
                c_boot,
            )

            delta = score_c - score_b

            if np.isfinite(delta):
                bootstrap_deltas.append(delta)

        except ValueError:
            # Can occur for metrics such as ROC-AUC if a
            # bootstrap sample contains only one class.
            continue

    bootstrap_deltas = np.asarray(
        bootstrap_deltas,
        dtype=float,
    )

    assert len(bootstrap_deltas) > 0

    ci_lower = np.percentile(
        bootstrap_deltas,
        2.5,
    )

    ci_upper = np.percentile(
        bootstrap_deltas,
        97.5,
    )

    return {
        "delta": float(observed_delta),
        "ci_lower": float(ci_lower),
        "ci_upper": float(ci_upper),
        "significant": bool(
            ci_lower > 0
            or ci_upper < 0
        ),
        "n_bootstrap": int(
            len(bootstrap_deltas)
        ),
    }

## 17 — Statistical comparison

The primary comparison is **C vs. B** — does ASR-native uncertainty add anything beyond
intent-native uncertainty alone? Not "does C beat A" (it almost certainly will).

Paired bootstrap confidence intervals on the ROC-AUC / PR-AUC / Brier deltas, matching
the methodology used for the Notebook 15 combined-vs-proxy comparison.

In [40]:
# ============================================================
# 17 — Statistical Comparison
# ============================================================
#
# IMPORTANT:
# Model-family selection must NOT use the FSC test set.
#
# Protocol:
#
#   FSC validation
#       ↓
#   5-fold CV model-family selection
#       ↓
#   freeze selected family
#       ↓
#   refit on full validation
#       ↓
#   ONE final FSC test evaluation
#
# Primary comparison:
#       C (ASR + Intent)
#       vs
#       B (Intent only)
#
# This avoids selecting the risk-model family using test ROC-AUC.
# ============================================================

from sklearn.model_selection import StratifiedKFold
from sklearn.base import clone


def select_risk_model_family_cv(
    feature_df: pd.DataFrame,
    target: pd.Series,
    train_idx,
    experiment_name: str,
    metric: str = "roc_auc",
    n_splits: int = 5,
) -> dict:
    """
    Select the risk-model family using ONLY the FSC validation split.

    Five-fold stratified CV is performed inside validation.

    Returns:
        {
            "selected_model": ...,
            "cv_results": ...
        }
    """

    feature_cols = EXPERIMENTS[experiment_name]

    X = feature_df.loc[
        train_idx,
        feature_cols,
    ].reset_index(drop=True)

    y = target.loc[
        train_idx
    ].reset_index(drop=True)

    skf = StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=RANDOM_SEED,
    )

    rows = []

    for model_name, model_fn in RISK_MODELS.items():

        fold_scores = []

        for fold_id, (fold_train, fold_valid) in enumerate(
            skf.split(X, y),
            start=1,
        ):

            model = model_fn()

            model.fit(
                X.iloc[fold_train],
                y.iloc[fold_train],
            )

            probs = model.predict_proba(
                X.iloc[fold_valid]
            )[:, 1]

            score = roc_auc_score(
                y.iloc[fold_valid],
                probs,
            )

            fold_scores.append(score)

            rows.append(
                {
                    "experiment": experiment_name,
                    "model": model_name,
                    "fold": fold_id,
                    "roc_auc": score,
                }
            )

        print(
            f"{experiment_name} | "
            f"{model_name} | "
            f"CV ROC-AUC: "
            f"{np.mean(fold_scores):.4f} "
            f"+/- "
            f"{np.std(fold_scores, ddof=1):.4f}"
        )

    cv_df = pd.DataFrame(rows)

    summary = (
        cv_df
        .groupby("model")["roc_auc"]
        .agg(
            mean="mean",
            std="std",
        )
        .sort_values(
            "mean",
            ascending=False,
        )
    )

    selected_model_name = summary.index[0]

    print()
    print(
        f"Selected risk-model family "
        f"for {experiment_name}: "
        f"{selected_model_name}"
    )

    return {
        "selected_model": selected_model_name,
        "cv_results": cv_df,
        "cv_summary": summary,
    }


if full_feature_df is not None:

    train_idx = full_feature_df.index[
        full_feature_df["split"] == "validation"
    ]

    test_idx = full_feature_df.index[
        full_feature_df["split"] == "test"
    ]

    y_all = full_feature_df[
        "intent_failed"
    ]

    # --------------------------------------------------------
    # Select one model family based on C validation CV.
    #
    # C is the richer model and is the primary model family
    # selection target. The same selected family is then used
    # for the B vs C head-to-head comparison.
    # --------------------------------------------------------

    model_selection = select_risk_model_family_cv(
        feature_df=full_feature_df,
        target=y_all,
        train_idx=train_idx,
        experiment_name="C_combined",
        n_splits=5,
    )

    best_model_name = model_selection[
        "selected_model"
    ]

    print()
    print("=" * 80)
    print("RISK MODEL FAMILY SELECTION")
    print("=" * 80)

    print(
        model_selection["cv_summary"]
        .to_string()
    )

    print()
    print(
        f"FROZEN MODEL FAMILY: "
        f"{best_model_name}"
    )


    X_train_b = full_feature_df.loc[
        train_idx,
        INTENT_FEATURES,
    ]

    X_test_b = full_feature_df.loc[
        test_idx,
        INTENT_FEATURES,
    ]

    X_train_c = full_feature_df.loc[
        train_idx,
        ASR_FEATURES + INTENT_FEATURES,
    ]

    X_test_c = full_feature_df.loc[
        test_idx,
        ASR_FEATURES + INTENT_FEATURES,
    ]

    y_train = full_feature_df.loc[
        train_idx,
        "intent_failed",
    ]

    y_test = full_feature_df.loc[
        test_idx,
        "intent_failed",
    ]

    model_b = (
        RISK_MODELS[best_model_name]()
        .fit(
            X_train_b,
            y_train,
        )
    )

    model_c = (
        RISK_MODELS[best_model_name]()
        .fit(
            X_train_c,
            y_train,
        )
    )

    probs_B = model_b.predict_proba(
        X_test_b
    )[:, 1]

    probs_C = model_c.predict_proba(
        X_test_c
    )[:, 1]

    c_vs_b = {

        "roc_auc": paired_bootstrap_delta(
            y_test,
            probs_B,
            probs_C,
            roc_auc_score,
        ),

        "pr_auc": paired_bootstrap_delta(
            y_test,
            probs_B,
            probs_C,
            average_precision_score,
        ),

        "brier": paired_bootstrap_delta(
            y_test,
            probs_B,
            probs_C,
            lambda yt, p: -brier_score_loss(
                yt,
                p,
            ),
        ),
    }

    print()
    print("=" * 80)
    print("FINAL FROZEN TEST — C vs B")
    print("=" * 80)

    print(
        f"Model family: {best_model_name}"
    )

    for metric, result in c_vs_b.items():

        print(
            f"Δ{metric}: "
            f"{result['delta']:+.4f} "
            f"95% CI "
            f"["
            f"{result['ci_lower']:+.4f}, "
            f"{result['ci_upper']:+.4f}"
            f"] "
            f"{'SIGNIFICANT' if result['significant'] else 'not significant'}"
        )

    predictions_df = full_feature_df.loc[
        test_idx,
        ["intent_failed"],
    ].copy()

    predictions_df[
        "prob_failed_B_intent_only"
    ] = probs_B

    predictions_df[
        "prob_failed_C_combined"
    ] = probs_C

    predictions_df.to_csv(
        PREDICTIONS_PATH,
        index=False,
    )
    if hasattr(
        model_c,
        "feature_importances_",
    ):

        importance_df = pd.DataFrame(
            {
                "feature": (
                    ASR_FEATURES
                    + INTENT_FEATURES
                ),
                "importance": (
                    model_c.feature_importances_
                ),
            }
        ).sort_values(
            "importance",
            ascending=False,
        )

    elif hasattr(
        model_c,
        "coef_",
    ):

        importance_df = pd.DataFrame(
            {
                "feature": (
                    ASR_FEATURES
                    + INTENT_FEATURES
                ),
                "importance": (
                    model_c.coef_[0]
                ),
            }
        ).sort_values(
            "importance",
            key=abs,
            ascending=False,
        )

    else:

        importance_df = None

    if importance_df is not None:

        importance_df.to_csv(
            IMPORTANCE_PATH,
            index=False,
        )

        print()
        print(
            "Final C model feature importance:"
        )

        print(
            importance_df.to_string(
                index=False
            )
        )

    print()
    print(
        f"Saved: {PREDICTIONS_PATH}"
    )

    print(
        f"Saved: {IMPORTANCE_PATH}"
    )

else:

    c_vs_b = None
    model_selection = None
    best_model_name = None

    print(
        "Statistical comparison skipped — "
        "run Sections 12–16 first."
    )

C_combined | logistic_regression | CV ROC-AUC: 0.7818 +/- 0.0453
C_combined | random_forest | CV ROC-AUC: 0.8438 +/- 0.0499

Selected risk-model family for C_combined: random_forest

RISK MODEL FAMILY SELECTION
                         mean       std
model                                  
random_forest        0.843753  0.049855
logistic_regression  0.781773  0.045334

FROZEN MODEL FAMILY: random_forest

FINAL FROZEN TEST — C vs B
Model family: random_forest
Δroc_auc: -0.0247 95% CI [-0.1441, +0.0926] not significant
Δpr_auc: -0.0398 95% CI [-0.1453, +0.0531] not significant
Δbrier: +0.0005 95% CI [-0.0005, +0.0014] not significant

Final C model feature importance:
              feature  importance
     asr_mean_entropy    0.118007
  asr_mean_confidence    0.117008
        intent_margin    0.100387
       intent_entropy    0.099627
    intent_confidence    0.089801
      asr_std_entropy    0.087827
asr_median_confidence    0.086408
   asr_std_confidence    0.079396
      asr_max_entro

## 18 — Cross-dataset comparison

The payoff cell: SLURP (Notebook 16, provisional pending its fixed-split rerun) side by
side with FSC (this notebook).

In [36]:
# ============================================================
# 18 — Cross-Dataset Comparison
# ============================================================
#
# FSC values come from the FROZEN risk-model family selected
# using validation-only CV in Section 17.
#
# We do NOT take max(test ROC-AUC) across model families.
# ============================================================

SLURP_PROVISIONAL = {
    "asr_native_roc_auc": 0.618,
    "intent_native_roc_auc": 0.911,
    "combined_roc_auc": 0.883,
}


if (
    risk_results_df is not None
    and best_model_name is not None
):

    # --------------------------------------------------------
    # Refit/evaluate the frozen selected family for A/B/C.
    #
    # This is purely for reporting the three arms using the
    # SAME pre-selected model family.
    # --------------------------------------------------------

    selected_results = []

    for experiment_name, feature_cols in EXPERIMENTS.items():

        X_train = full_feature_df.loc[
            train_idx,
            feature_cols,
        ]

        X_test = full_feature_df.loc[
            test_idx,
            feature_cols,
        ]

        y_train = full_feature_df.loc[
            train_idx,
            "intent_failed",
        ]

        y_test = full_feature_df.loc[
            test_idx,
            "intent_failed",
        ]

        model = (
            RISK_MODELS[best_model_name]()
            .fit(
                X_train,
                y_train,
            )
        )

        probs = model.predict_proba(
            X_test
        )[:, 1]

        selected_results.append(
            {
                "experiment": experiment_name,
                "roc_auc": roc_auc_score(
                    y_test,
                    probs,
                ),
            }
        )

    selected_results_df = pd.DataFrame(
        selected_results
    )

    fsc_roc_auc = {
        "asr_native_roc_auc": float(
            selected_results_df.loc[
                selected_results_df["experiment"]
                == "A_asr_only",
                "roc_auc",
            ].iloc[0]
        ),

        "intent_native_roc_auc": float(
            selected_results_df.loc[
                selected_results_df["experiment"]
                == "B_intent_only",
                "roc_auc",
            ].iloc[0]
        ),

        "combined_roc_auc": float(
            selected_results_df.loc[
                selected_results_df["experiment"]
                == "C_combined",
                "roc_auc",
            ].iloc[0]
        ),
    }

    cross_dataset_df = pd.DataFrame(
        {
            "experiment": [
                "ASR-native",
                "Intent-native",
                "Combined",
                "Combined - Intent",
            ],

            "SLURP": [
                SLURP_PROVISIONAL[
                    "asr_native_roc_auc"
                ],

                SLURP_PROVISIONAL[
                    "intent_native_roc_auc"
                ],

                SLURP_PROVISIONAL[
                    "combined_roc_auc"
                ],

                (
                    SLURP_PROVISIONAL[
                        "combined_roc_auc"
                    ]
                    -
                    SLURP_PROVISIONAL[
                        "intent_native_roc_auc"
                    ]
                ),
            ],

            "FSC": [
                fsc_roc_auc[
                    "asr_native_roc_auc"
                ],

                fsc_roc_auc[
                    "intent_native_roc_auc"
                ],

                fsc_roc_auc[
                    "combined_roc_auc"
                ],

                (
                    fsc_roc_auc[
                        "combined_roc_auc"
                    ]
                    -
                    fsc_roc_auc[
                        "intent_native_roc_auc"
                    ]
                ),
            ],
        }
    )

    cross_dataset_df.to_csv(
        CROSS_DATASET_PATH,
        index=False,
    )

    print(
        cross_dataset_df.to_string(
            index=False
        )
    )

    print()
    print(
        f"Frozen FSC risk-model family: "
        f"{best_model_name}"
    )

    print(
        "(SLURP values are provisional until "
        "Notebook 16's fixed-split rerun.)"
    )

else:

    fsc_roc_auc = None

    print(
        "Cross-dataset table requires "
        "the frozen Section 17 result."
    )

       experiment  SLURP       FSC
       ASR-native  0.618  0.695139
    Intent-native  0.911  0.735325
         Combined  0.883  0.710610
Combined - Intent -0.028 -0.024715

Frozen FSC risk-model family: random_forest
(SLURP values are provisional until Notebook 16's fixed-split rerun.)


In [37]:
# --- Pattern classification -------------------------------------------------
# Pattern A: SLURP C<B, FSC C>B  -> complementarity is dataset-dependent
# Pattern B: SLURP C<B, FSC C<B  -> current ASR uncertainty adds little beyond intent
# Pattern C: SLURP C>B, FSC C>B  -> strong support for H1
# Pattern D: mixed statistical evidence -> requires interpretation, not a forced conclusion

def classify_pattern(slurp_combined_beats_intent: bool, fsc_combined_beats_intent: bool,
                      fsc_ci_lower: float) -> str:
    if not slurp_combined_beats_intent and fsc_combined_beats_intent and fsc_ci_lower > 0:
        return "Pattern A — complementarity is dataset-dependent"
    if not slurp_combined_beats_intent and not fsc_combined_beats_intent:
        return "Pattern B — ASR uncertainty adds little beyond intent uncertainty"
    if slurp_combined_beats_intent and fsc_combined_beats_intent and fsc_ci_lower > 0:
        return "Pattern C — strong support for H1"
    return "Pattern D — mixed evidence, requires interpretation"

def final_verdict(combined_delta: float, ci_lower: float) -> str:
    if combined_delta > 0 and ci_lower > 0:
        return "H1 SUPPORTED"
    elif combined_delta < 0:
        return "H1 NOT SUPPORTED"
    else:
        return "H1 INCONCLUSIVE"


if c_vs_b is not None and fsc_roc_auc is not None:
    verdict = final_verdict(c_vs_b["roc_auc"]["delta"], c_vs_b["roc_auc"]["ci_lower"])
    pattern = classify_pattern(
        slurp_combined_beats_intent=SLURP_PROVISIONAL["combined_roc_auc"] > SLURP_PROVISIONAL["intent_native_roc_auc"],
        fsc_combined_beats_intent=fsc_roc_auc["combined_roc_auc"] > fsc_roc_auc["intent_native_roc_auc"],
        fsc_ci_lower=c_vs_b["roc_auc"]["ci_lower"],
    )

    print("=" * 50)
    print("NOTEBOOK 17 — CROSS-DATASET H1 VERDICT")
    print("=" * 50)
    print(f"Dataset:            Fluent Speech Commands")
    print(f"Evaluation split:   TEST")
    print(f"Examples:           {len(test_df)}")
    print()
    print(f"Intent-only ROC-AUC:  {fsc_roc_auc['intent_native_roc_auc']:.4f}")
    print(f"Combined ROC-AUC:     {fsc_roc_auc['combined_roc_auc']:.4f}")
    print(f"Delta (C - B):        {c_vs_b['roc_auc']['delta']:+.4f}")
    print(f"95% CI:               [{c_vs_b['roc_auc']['ci_lower']:+.4f}, {c_vs_b['roc_auc']['ci_upper']:+.4f}]")
    print()
    print(f"Pattern:  {pattern}")
    print(f"H1:       {verdict}")
    print("=" * 50)
    print()
    print("Reminder: SLURP values above are provisional (Notebook 16 pending its "
          "fixed-split rerun) — don't promote this verdict to the README/paper/"
          "abstract/conclusion until that rerun lands (see Section 18).")
else:
    verdict = None
    pattern = None
    print("Verdict cell defined — runs once Sections 07/09 checkpoints and Section 16/17 results exist.")


NOTEBOOK 17 — CROSS-DATASET H1 VERDICT
Dataset:            Fluent Speech Commands
Evaluation split:   TEST
Examples:           3793

Intent-only ROC-AUC:  0.7353
Combined ROC-AUC:     0.7106
Delta (C - B):        -0.0247
95% CI:               [-0.1433, +0.0896]

Pattern:  Pattern B — ASR uncertainty adds little beyond intent uncertainty
H1:       H1 NOT SUPPORTED

Reminder: SLURP values above are provisional (Notebook 16 pending its fixed-split rerun) — don't promote this verdict to the README/paper/abstract/conclusion until that rerun lands (see Section 18).


## Artifacts

    reports/
    ├── fsc_dataset_audit.csv
    ├── fsc_asr_predictions.csv
    ├── fsc_intent_predictions.csv
    ├── fsc_voxintel_r_features.csv
    ├── fsc_voxintel_r_model_comparison.csv
    ├── fsc_voxintel_r_predictions.csv
    ├── fsc_voxintel_r_feature_importance.csv
    ├── fsc_voxintel_r_cross_dataset_comparison.csv
    └── fsc_voxintel_r_summary.json

    models/
    ├── wav2vec2_fsc/
    └── distilbert_fsc_intent/

### Split policy (Section 30 of the design discussion)

The FSC test set is **not** used to train the risk model:

    FSC TRAIN        → ASR Transformer + Intent Transformer
    FSC VALIDATION    → checkpoint selection + risk-model development
    FSC TEST          → final, frozen VoxIntel-R evaluation only

### What happens next

Do **not** modify the finalized README's roadmap yet. Run this notebook to an actual
result first. Only then update the roadmap from

    16 → 17 calibration → 18 cost

to the experimentally justified

    16 → 17 cross-dataset validation → 18 calibration → 19 cost

so the README stays honest and the conclusion isn't written before the experiment runs.

In [38]:
if full_feature_df is None:
    status = ("scaffold — checkpoints not trained yet. Set RUN_ASR_TRAINING / "
               "RUN_INTENT_TRAINING = True in Section 03, or point FSC_ASR_MODEL_DIR / "
               "FSC_INTENT_MODEL_DIR at existing checkpoints, then rerun.")
elif risk_results_df is None:
    status = "features built — risk experiments (Section 16) not yet run"
elif c_vs_b is None:
    status = "risk experiments run — statistical comparison (Section 17) pending"
else:
    status = f"complete — {verdict} ({pattern})"

summary = {
    "notebook_id": NOTEBOOK_ID,
    "dataset": DATASET,
    "hypothesis": HYPOTHESIS,
    "random_seed": RANDOM_SEED,
    "asr_features": ASR_FEATURES,
    "intent_features": INTENT_FEATURES,
    "forbidden_features_checked": sorted(FORBIDDEN_FEATURES),
    "asr_checkpoint_trained": _dir_has_checkpoint(FSC_ASR_MODEL_DIR),
    "intent_checkpoint_trained": _dir_has_checkpoint(FSC_INTENT_MODEL_DIR),
    "feature_matrix_built": full_feature_df is not None,
    "n_validation_rows": int((full_feature_df["split"] == "validation").sum()) if full_feature_df is not None else None,
    "n_test_rows": int((full_feature_df["split"] == "test").sum()) if full_feature_df is not None else None,
    "risk_experiments_run": risk_results_df is not None,
    "fsc_roc_auc": {k: float(v) for k, v in fsc_roc_auc.items()} if fsc_roc_auc is not None else None,
    "c_vs_b_delta_roc_auc": float(c_vs_b["roc_auc"]["delta"]) if c_vs_b is not None else None,
    "c_vs_b_ci": [float(c_vs_b["roc_auc"]["ci_lower"]), float(c_vs_b["roc_auc"]["ci_upper"])] if c_vs_b is not None else None,
    "h1_verdict": verdict,
    "pattern": pattern,
    "status": status,
}

with open(SUMMARY_PATH, "w") as f:
    json.dump(summary, f, indent=2, default=str)

print(f"Saved: {SUMMARY_PATH}")
print(json.dumps(summary, indent=2, default=str))


Saved: C:\Users\ACER\OneDrive\Desktop\VoxIntel\reports\fsc_voxintel_r_summary.json
{
  "notebook_id": "17",
  "dataset": "FSC",
  "hypothesis": "H1",
  "random_seed": 42,
  "asr_features": [
    "asr_mean_confidence",
    "asr_min_confidence",
    "asr_std_confidence",
    "asr_median_confidence",
    "asr_mean_entropy",
    "asr_max_entropy",
    "asr_std_entropy",
    "asr_num_frames",
    "audio_duration"
  ],
  "intent_features": [
    "intent_confidence",
    "intent_entropy",
    "intent_margin"
  ],
  "forbidden_features_checked": [
    "cer",
    "error_type",
    "ground_truth_intent",
    "lexical_overlap",
    "reference_transcript",
    "taxonomy",
    "wer"
  ],
  "asr_checkpoint_trained": true,
  "intent_checkpoint_trained": true,
  "feature_matrix_built": true,
  "n_validation_rows": 3118,
  "n_test_rows": 3793,
  "risk_experiments_run": true,
  "fsc_roc_auc": {
    "asr_native_roc_auc": 0.69513896897705,
    "intent_native_roc_auc": 0.7353246453696229,
    "combined_roc

## Conclusion

External validation on Fluent Speech Commands (FSC), evaluated under FSC's own train→validation→test contract (never pooled with SLURP), tested whether reference-free ASR-native uncertainty adds predictive signal beyond intent-native uncertainty. It does not — replicating the SLURP finding on an independently trained ASR + intent Transformer stack.

**Key results** (frozen FSC test; risk-model family = RandomForest, selected by validation-only CV)
- ROC-AUC — ASR-native / intent-native / combined: **0.695 / 0.735 / 0.711**.
- Combined − intent-native ROC-AUC: **−0.0247**, paired-bootstrap 95% CI **[−0.1433, +0.0896]** — not significant (spans zero).
- **H1 NOT SUPPORTED** (Pattern B — ASR uncertainty adds little beyond intent uncertainty).

**Caveat:** FSC is nearly saturated for this stack — test WER 1.8%, clean-text intent accuracy 100%, downstream failure prevalence only 0.61% (~23 failures / 3,793) — so the C-vs-B comparison has low statistical power and correspondingly wide CIs; this null is weak evidence, not proof of no complementarity. SLURP values in the cross-dataset table remain provisional pending Notebook 16's fixed-split rerun.

**Feeds into:** the calibration + selective-prediction notebook (18) and the phase 8 hypothesis-validation reports, which consume this FSC external-validation result.